# HALO dashboard: run the whole chain for one site

This notebook runs the full chain for one site config: validate, pipeline, statistics, checks and the static dashboard. It only calls functions of `core/` and `scripts/` and shows the tables they wrote. It computes no statistic, fit, match or accuracy number itself, so it cannot drift away from the pipeline.

To run another site, change `CONFIG` in step 1 and nothing else. `FORCE = True` recomputes every cached table (slow, it reads the point clouds again). With `FORCE = False` a stage only runs again when its config sections or input files changed.

Run the cells from top to bottom in one kernel. The pipeline must run in one process.

## 1. Settings
Set the config path (relative to the repository root), `FORCE` and `HEIGHT_VARIANTS`. `HEIGHT_VARIANTS = True` adds the height-variant analysis, which needs one extra pass over the cloud. The cell finds the repository root, loads the config and defines two small display helpers. Check the printed site id and folders.

In [ ]:
CONFIG = "config/site_155.yaml"   # relative to the repository root
FORCE = False                     # True: recompute every cached table (slow)
HEIGHT_VARIANTS = False           # True: run the height-variant analysis (one extra pass over the cloud)

import json, sys, time
from pathlib import Path
import pandas as pd

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "core").is_dir() and (p / "scripts").is_dir())
for p in (ROOT, ROOT / "scripts"):
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))

import run_pipeline, run_stats, build_dashboard
from core import cache, export, validate
from core.config import load_config
from core.tables import has_table, read_table

CONFIG_PATH = ROOT / CONFIG
cfg = load_config(CONFIG_PATH)
pd.set_option("display.max_columns", 40, "display.width", 200, "display.max_rows", 80)

def table_or_none(name):
    """Return a table, or None (with a note) when it is not available for this site."""
    if not has_table(cfg, name):
        print(f"{name}: not available")
        return None
    return read_table(cfg, name)

def value_counts_of(name, column):
    """Show the counts of one column of a table; say so if table or column is missing."""
    df = table_or_none(name)
    if df is None:
        return
    if column not in df.columns:
        print(f"{name}: column '{column}' not available")
        return
    display(df[column].value_counts(dropna=False).rename_axis(column).to_frame("rows"))

print("repository root:", ROOT)
print("site:", cfg.site.id, "|", cfg.site.name)
print("output folder:", cfg.output_dir)
print("tables folder:", cfg.tables_dir)

## 2. Validate the configuration
`core.validate.validate` checks the config against its schema and checks that the input files and columns exist. It returns a list of problems. The list should be empty. If not, the notebook stops here.

In [ ]:
problems = validate.validate(CONFIG_PATH)
if problems:
    for line in problems:
        print(line)
    raise RuntimeError(f"Config {CONFIG} is invalid: {len(problems)} problems (listed above).")
print("Config is valid.")

## 3. Run the pipeline
`core.cache.set_force` sets the recompute switch. Then every stage in `run_pipeline.STAGES` is run with `run_pipeline.run_stage`. The table shows seconds and status per stage. A cached stage that is up to date takes almost no time. Some stages are not cached and read the point cloud on every run, so this cell can take several minutes. At the first failure the notebook stops, so that later cells never run on stale tables.

In [ ]:
cache.set_force(FORCE)
timing, failure = [], None
for name, module in run_pipeline.STAGES:
    t0 = time.time()
    try:
        run_pipeline.run_stage(name, module, cfg)
        status = "ok"
    except Exception as exc:
        status, failure = f"FAILED: {type(exc).__name__}: {exc}", exc
    timing.append({"stage": name, "module": module, "seconds": round(time.time() - t0, 1), "status": status})
    if failure is not None:
        break
display(pd.DataFrame(timing))
if failure is not None:
    raise failure

## 4a. Inventory
`field_trees.csv` is written by the inventory stage. The cell shows the number of trees, the trees per species and how many trees have a sample height (`has_height`).

In [ ]:
ft = table_or_none("field_trees.csv")
if ft is not None:
    print("trees:", len(ft))
    if "has_height" in ft.columns:
        print("trees with a sample height:", int(ft["has_height"].sum()))
    value_counts_of("field_trees.csv", "species")

## 4b. Orientation check
`orientation_check.csv` (stage orientation) counts, per plot, how many trees fit the plot corners for each mirroring variant. The column `best` should agree with the mirrored plots set in the config.

In [ ]:
oc = table_or_none("orientation_check.csv")
if oc is not None:
    display(oc)

## 4c. Detection classes
Stage matching writes `field_trees_detection.csv` (one row per field tree, column `detection`) and `detection_per_parzelle.csv`. Look at how many trees are detected, uncertain or not detected, overall and per plot.

In [ ]:
value_counts_of("field_trees_detection.csv", "detection")
dp = table_or_none("detection_per_parzelle.csv")
if dp is not None:
    display(dp)

## 4d. Result tiers
`match_tiers.csv` (stage tiers) says by which method each tree was matched. Manual and refined tiers are separate from the automatic result. Look at the counts of `method`.

In [ ]:
value_counts_of("match_tiers.csv", "method")

## 4e. Unmatched LiDAR instances
`lidar_instances_without_field_tree.csv` (stage unmatched) lists laser instances that no field tree was matched to. The cell shows the row count and the first rows.

In [ ]:
um = table_or_none("lidar_instances_without_field_tree.csv")
if um is not None:
    print("unmatched LiDAR instances:", len(um))
    display(um.head())

## 4f. Identity classes
`match_identity.csv` (stage identity) gives each match an identity class (confirmed, probable, ambiguous, rejected, unmatched). Look at the counts.

In [ ]:
value_counts_of("match_identity.csv", "identity_class")

## 4g. LiDAR stem diameter
`height_dbh_all_trees.csv` (stage dbh) holds the stem diameter measured in the point cloud. The cell shows the row count, the first rows and the quality classes.

In [ ]:
dbh = table_or_none("height_dbh_all_trees.csv")
if dbh is not None:
    print("rows:", len(dbh))
    display(dbh.head())
    value_counts_of("height_dbh_all_trees.csv", "dbh_quality")

## 4h. Volume tables
Stage volume writes the form-factor volumes, stage volume_gri the volumes by the GRI method, stage increment the increment. A table that does not exist for this site is reported as not available.

In [ ]:
for name in ["volume_per_parzelle.csv", "volume_per_species.csv", "volume_gri_per_parzelle.csv",
             "volume_gri_per_species.csv", "increment_measured.csv", "increment_by_age.csv"]:
    df = table_or_none(name)
    if df is not None:
        print(f"{name}: {len(df)} rows")
        display(df.head())

## 5. Statistics
`run_stats.run(cfg, write_tables=True, ...)` computes all statistics, writes `stats.json` and the analysis tables the dashboard needs (for example `sample_trees.csv`). The tables below only show what is in `stats.json`. Ids that exist for both subsets `all_matched` and `detected_only` are shown side by side. The sentences follow, grouped by the first part of the id.

In [ ]:
stats_path = run_stats.run(cfg, write_tables=True, height_variants=HEIGHT_VARIANTS)
stats_df = pd.DataFrame(json.loads(Path(stats_path).read_text(encoding="utf-8")))
print("stats.json:", stats_path)
print("entries:", len(stats_df))
cols = ["id", "subset", "value", "unit", "n", "ci_low", "ci_high"]
display(stats_df[cols])

In [ ]:
both = stats_df[stats_df["subset"].isin(["all_matched", "detected_only"])]
if both.empty:
    print("not available: no entries with subsets all_matched / detected_only")
else:
    side = both.pivot_table(index="id", columns="subset", values="value", aggfunc="first")
    with pd.option_context("display.max_rows", None):   # show every row, not a shortened view
        display(side.dropna())                           # only ids that exist for both subsets

In [ ]:
stats_df["family"] = stats_df["id"].str.split(".").str[0]
for family, group in stats_df.groupby("family", sort=False):
    print(f"\n== {family} ({len(group)})")
    for sentence in group["statement"].dropna():
        print("-", sentence)

## 6. Checks before the build
The dashboard needs the tables in `core.export.REQUIRED`. Optional tables that are missing are reported by `core.export.build_data`. Sample-tree images are looked up in the folder set in the config key `dashboard.sample_figures`. The table shows ok, missing, or found n of m expected images. A missing required table stops the notebook.

In [ ]:
rows = [{"check": f"required: {n}", "status": "ok" if has_table(cfg, n) else "missing"} for n in export.REQUIRED]
absent = [r["check"] for r in rows if r["status"] != "ok"]
if absent:
    display(pd.DataFrame(rows))
    raise FileNotFoundError("Required tables missing: " + ", ".join(absent))
data, figs = export.build_data(cfg, None, Path(cfg.output_dir))
missing_opt = data["meta"]["missing_tables"]
rows.append({"check": "optional tables missing", "status": ", ".join(missing_opt) if missing_opt else "ok"})
sf = cfg.get("dashboard", None)
sf = sf.get("sample_figures", None) if sf is not None else None
if sf is None:
    rows.append({"check": "sample-tree images", "status": "not configured"})
else:
    expected = len(data["samples"]) * (1 + len(sf.get("extra_patterns", []) or []))
    rows.append({"check": "sample-tree images", "status": f"{len(figs)} of {expected}"})
display(pd.DataFrame(rows))

## 7. Build the dashboard
`build_dashboard.build` writes `index.html`, `data.js`, the scripts and the sample images into `dist/<site_id>/`. The page opens by double-click, no web server is needed. The cell lists the files with their sizes.

In [ ]:
out = Path(build_dashboard.build(CONFIG_PATH))
print("open this file:", out / "index.html")
files = sorted(p for p in out.rglob("*") if p.is_file())
display(pd.DataFrame({"file": [str(p.relative_to(out)) for p in files],
                      "size_kb": [round(p.stat().st_size / 1024, 1) for p in files]}))

## 8. Share this notebook as HTML
Run all cells, save the notebook (Ctrl+S), then run this cell. It exports the notebook file as saved on disk to `notebook.html` in the dashboard folder. The HTML file contains all tables and can be sent to colleagues. Git stores the notebook itself without outputs.

In [ ]:
import nbformat
saved = nbformat.read(ROOT / "notebooks" / "04_dashboard.ipynb", as_version=4)
has_outputs = any(c.cell_type == "code" and c.get("outputs") for c in saved.cells)
try:
    from nbconvert import HTMLExporter
except ImportError:
    HTMLExporter = None
    print("nbconvert is not installed; nothing was exported.")
if HTMLExporter is not None:
    if not has_outputs:
        print("Save the notebook first (Ctrl+S), then run this cell again; nothing was exported.")
    else:
        body, _ = HTMLExporter().from_notebook_node(saved)
        html_path = out / "notebook.html"
        html_path.write_text(body, encoding="utf-8")
        print("written:", html_path)